# Regularization with SciKit-Learn

Regularization attempts to minimize the RSS (residual sum of squares) *and* a penalty factor. This penalty factor will penalize models that have coefficients that are too large. Some methods of regularization will actually cause non useful features to have a coefficient of zero, in which case the model does not consider the feature.

Let's explore two methods of regularization, Ridge Regression and Lasso. We'll combine these with the polynomial feature set (it wouldn't be as effective to perform regularization of a model on such a small original feature set of the original X).

---
## Imports

In [64]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

---
## Data and Setup

In [65]:
df = pd.read_csv("../../data/Advertising.csv")
X = df.drop('sales', axis=1)
y = df['sales']

---
### Polynomial Conversion

In [66]:
from sklearn.preprocessing import PolynomialFeatures

In [67]:
polynomial_converter = PolynomialFeatures(degree=3, include_bias=False)

In [68]:
X_poly = polynomial_converter.fit_transform(X)

---
### Train | Test Split

In [69]:
from sklearn.model_selection import train_test_split

In [70]:
X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.3, random_state=101)

---
## Scaling the Data

While our particular data set has all the values in the same order of magnitude ($1000s of dollars spent), typically that won't be the case on a dataset, and since the mathematics behind regularized models will sum coefficients together, its important to standardize the features. Review the theory videos for more info, as well as a discussion on why we only **fit** to the training data, and **transform** on both sets separately.

In [71]:
from sklearn.preprocessing import StandardScaler

In [72]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) # here will calculate mean and std for scaling and transform the X_train
X_test = scaler.transform(X_test) # here will only transform X_test using the mean and std of X_train

---
## Ridge Regression

In [73]:
from sklearn.linear_model import Ridge

In [74]:
ridge_model = Ridge(alpha=10)

In [75]:
ridge_model.fit(X_train, y_train)

,alpha,10
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [76]:
test_predictions = ridge_model.predict(X_test)

In [77]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [78]:
MAE = mean_absolute_error(y_test, test_predictions)
MSE = mean_squared_error(y_test, test_predictions)
RMSE = np.sqrt(MSE)

In [79]:
print(f"Ridge Regression MAE: {MAE}")
print(f"Ridge Regression MSE: {MSE}")
print(f"Ridge Regression RMSE: {RMSE}")

Ridge Regression MAE: 0.5774404204714163
Ridge Regression MSE: 0.8003783071528352
Ridge Regression RMSE: 0.894638646131965


---
### Choosing an alpha value with Cross-Validation

In [80]:
from sklearn.linear_model import RidgeCV

In [81]:
ridge_cv_model = RidgeCV(alphas=(0.1, 1.0, 10.0), scoring='neg_mean_absolute_error')  
# Here we use neg_mean_absolute_error as our metric because RidgeCV maximize the score, so the maximum of negative MAE is the minimum of MAE
# NOTE: the cv parameter can also be set to specify the number of folds in cross-validation, but it using leave-one-out by default, which make computation longer for larger datasets

ridge_cv_model.fit(X_train, y_train) # here X_train and y_train will be splitted internally for cross-validation to X_val and y_val sets

,alphas,"(0.1, ...)"
,fit_intercept,True
,scoring,'neg_mean_absolute_error'
,cv,None
,gcv_mode,None
,store_cv_results,False
,alpha_per_target,False


In [82]:
# Best alpha value found
ridge_cv_model.alpha_

np.float64(0.1)

In [83]:
test_predictions = ridge_cv_model.predict(X_test)

In [84]:
MAE = mean_absolute_error(y_test, test_predictions)
MSE = mean_squared_error(y_test, test_predictions)
RMSE = np.sqrt(MSE)

In [85]:
print(f"RidgeCV Regression MAE: {MAE}")
print(f"RidgeCV Regression MSE: {MSE}")
print(f"RidgeCV Regression RMSE: {RMSE}")

RidgeCV Regression MAE: 0.427377488434534
RidgeCV Regression MSE: 0.3820129881525864
RidgeCV Regression RMSE: 0.6180719926938822


In [86]:
ridge_cv_model.coef_

array([ 5.40769392,  0.5885865 ,  0.40390395, -6.18263924,  4.59607939,
       -1.18789654, -1.15200458,  0.57837796, -0.1261586 ,  2.5569777 ,
       -1.38900471,  0.86059434,  0.72219553, -0.26129256,  0.17870787,
        0.44353612, -0.21362436, -0.04622473, -0.06441449])

---
## Lasso Regression - Least Absolute Shrinkage and Selection Operator

In [87]:
from sklearn.linear_model import LassoCV

In [98]:
lasso_cv_model = LassoCV(eps=0.1, cv=5, n_alphas=100,) # here we set cv=5 for 5-Fold Cross-Validation, eps is the length of the path, max_iter is the maximum number of iterations because Lasso may need more iterations to converge
# n_alphas is the number of alpha values to test

In [99]:
lasso_cv_model.fit(X_train, y_train)

/home/ahmed-hemdan/PycharmProjects/ML-Workshop/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:1622: FutureWarning: 'n_alphas' was deprecated in 1.7 and will be removed in 1.9. 'alphas' now accepts an integer value which removes the need to pass 'n_alphas'. The default value of 'alphas' will change from None to 100 in 1.9. Pass an explicit value to 'alphas' and leave 'n_alphas' to its default value to silence this warning.
  warnings.warn(


,eps,0.1
,n_alphas,100
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.0001
,copy_X,True
,cv,5
,verbose,False
,n_jobs,None


In [100]:
lasso_cv_model.alpha_

np.float64(0.4943070909225831)

In [101]:
test_predictions = lasso_cv_model.predict(X_test)

In [102]:
MAE = mean_absolute_error(y_test, test_predictions)
MSE = mean_squared_error(y_test, test_predictions)
RMSE = np.sqrt(MSE)

In [103]:
print(f"LassoCV Regression MAE: {MAE}")
print(f"LassoCV Regression MSE: {MSE}")
print(f"LassoCV Regression RMSE: {RMSE}")

LassoCV Regression MAE: 0.6541723161252868
LassoCV Regression MSE: 1.2787088713079886
LassoCV Regression RMSE: 1.1308001022762548


##### The benefit of Lasso is that it can perform feature selection by setting some coefficients to zero, as we see below.

In [104]:
lasso_cv_model.coef_

array([1.002651  , 0.        , 0.        , 0.        , 3.79745279,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        ])

---
## Elastic Net

Elastic Net combines the penalties of ridge regression and lasso in an attempt to get the best of both

In [105]:
from sklearn.linear_model import ElasticNetCV

In [114]:
elastic_net_model = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1], tol=0.01)
# Here we explain each parameter
# l1_ratio: The mixing parameter between Lasso (L1) and Ridge (L2) regularization. 0 corresponds to Ridge, 1 to Lasso.
# tol: The tolerance for the optimization.

In [115]:
elastic_net_model.fit(X_train, y_train)

,l1_ratio,"[0.1, 0.5, ...]"
,eps,0.001
,n_alphas,'deprecated'
,alphas,'warn'
,fit_intercept,True
,precompute,'auto'
,max_iter,1000
,tol,0.01
,cv,None
,copy_X,True
,verbose,0


##### We got a L1_Ratio of 1.0 which means the best model is actually Lasso in this case.

In [116]:
elastic_net_model.l1_ratio_

np.float64(1.0)

In [117]:
test_predictions = elastic_net_model.predict(X_test)

In [118]:
MAE = mean_absolute_error(y_test, test_predictions)
MSE = mean_squared_error(y_test, test_predictions)
RMSE = np.sqrt(MSE)

In [119]:
print(f"ElasticNetCV Regression MAE: {MAE}")
print(f"ElasticNetCV Regression MSE: {MSE}")
print(f"ElasticNetCV Regression RMSE: {RMSE}")

ElasticNetCV Regression MAE: 0.566326211756945
ElasticNetCV Regression MSE: 0.5603340214638839
ElasticNetCV Regression RMSE: 0.7485546215633726


In [120]:
elastic_net_model.coef_

array([ 3.78993643,  0.89232919,  0.28765395, -1.01843566,  2.15516144,
       -0.3567547 , -0.271502  ,  0.09741081,  0.        , -1.05563151,
        0.2362506 ,  0.07980911,  1.26170778,  0.01464706,  0.00462336,
       -0.39986069,  0.        ,  0.        , -0.05343757])